In [1]:
import sys; print(sys.executable)

/home/cjen/mySageMaker/ML/Presidio/dat/.venv/bin/python


In [2]:
import boto3

region = boto3.Session().region_name
s3 = boto3.client("s3", region_name=region)

In [3]:
from sagemaker import Session, s3

session = Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml


In [4]:
import boto3

boto_session = boto3.Session()

region  = boto_session.region_name
account = boto3.client("sts").get_caller_identity()["Account"]
bucket  = f"sagemaker-{region}-{account}"

# Data is already staged in S3 — no local upload needed
inputs = f"s3://{bucket}/spark-nlp/redaction/jgh-msss-prem/redacted_input"
print("S3 input location: " + inputs)

S3 input location: s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_input


In [5]:
import subprocess, sys, os

java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

Java already available: openjdk version "17.0.18" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [6]:
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "presidio-analyzer>=2.2" \
    "presidio-anonymizer>=2.2" \
    "spacy>=3.4" \
    "boto3" \
    "langdetect"

!python -m spacy download en_core_web_lg --quiet
!python -m spacy download fr_core_news_lg --quiet

print("All packages installed.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_lg')
All packages installed.


In [7]:
import sagemaker, os
from sagemaker import get_execution_role

sm_session = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region = boto_session.region_name or sm_session.boto_region_name
bucket = sm_session.default_bucket()
prefix = "presidio/redaction"

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Default S3 bucket     : {bucket}")
print(f"Execution role ARN    : {role}")

SageMaker SDK version : 2.253.1
Region                : us-west-2
Default S3 bucket     : sagemaker-us-west-2-493644444178
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd


In [8]:
# ── Verify this notebook is running within a SageMaker environment ─────────────
import os, json, urllib.request

print("=" * 60)
print("  SageMaker Session Proof")
print("=" * 60)

print(f"\n[SDK] sagemaker.__version__  : {sagemaker.__version__}")
print(f"[SDK] Session boto region    : {sm_session.boto_region_name}")
print(f"[SDK] Default S3 bucket      : {sm_session.default_bucket()}")
print(f"[SDK] Execution role ARN     : {role}")
print(f"[SDK] Boto3 caller identity  : {boto3.client('sts').get_caller_identity()['Arn']}")

sm_keys = sorted(k for k in os.environ if k.startswith("SM_") or "SAGEMAKER" in k.upper())
if sm_keys:
    print(f"\n[ENV] SageMaker env vars found ({len(sm_keys)}):")
    for k in sm_keys:
        print(f"      {k} = {os.environ[k]}")
else:
    print("\n[ENV] No SM_*/SAGEMAKER_* env vars — typical for Studio JupyterLab or local run.")

meta_path = "/opt/ml/metadata/resource-metadata.json"
if os.path.exists(meta_path):
    with open(meta_path) as fh:
        meta = json.load(fh)
    print(f"\n[META] {meta_path}:")
    print(json.dumps(meta, indent=4))
else:
    print(f"\n[META] {meta_path} not found (not inside a managed job container).")

_imds = "http://169.254.169.254/latest"
try:
    token = urllib.request.urlopen(
        urllib.request.Request(
            _imds + "/api/token",
            headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
            method="PUT",
        ), timeout=1
    ).read().decode()
    inst_type = urllib.request.urlopen(
        urllib.request.Request(
            _imds + "/meta-data/instance-type",
            headers={"X-aws-ec2-metadata-token": token},
        ), timeout=1
    ).read().decode()
    print(f"\n[IMDS] EC2 instance type : {inst_type}  ← confirms managed SageMaker host")
except Exception:
    print("\n[IMDS] EC2 IMDS not reachable — expected when running locally.")

print("\n✓ SageMaker SDK session is authenticated and active.")

  SageMaker Session Proof

[SDK] sagemaker.__version__  : 2.253.1
[SDK] Session boto region    : us-west-2
[SDK] Default S3 bucket      : sagemaker-us-west-2-493644444178
[SDK] Execution role ARN     : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd
[SDK] Boto3 caller identity  : arn:aws:sts::493644444178:assumed-role/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd/mindy@theclinician.com

[ENV] No SM_*/SAGEMAKER_* env vars — typical for Studio JupyterLab or local run.

[META] /opt/ml/metadata/resource-metadata.json not found (not inside a managed job container).

[IMDS] EC2 IMDS not reachable — expected when running locally.

✓ SageMaker SDK session is authenticated and active.


In [9]:
import sys
print("Python executable:", sys.executable)

from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import pyspark.sql.functions as F
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "en", "model_name": "en_core_web_lg"},
        {"lang_code": "fr", "model_name": "fr_core_news_lg"},
    ],
})
analyzer  = AnalyzerEngine(
    nlp_engine=provider.create_engine(),
    supported_languages=["en", "fr"],
)
anonymizer = AnonymizerEngine()
print("Presidio engines initialised (EN + FR).")

Python executable: /home/cjen/mySageMaker/ML/Presidio/dat/.venv/bin/python
Presidio engines initialised (EN + FR).


In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Presidio-Redaction")
    .config("spark.driver.memory",        "16G")
    .config("spark.driver.maxResultSize", "2000M")
    .config("spark.ui.enabled",           "false")
    .getOrCreate()
)
spark

26/03/31 10:06:49 WARN Utils: Your hostname, MindyJen1008 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/31 10:06:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 10:06:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Run Presidio on SageMaker via PySparkProcessor

The cells below use **`sagemaker.spark.processing.PySparkProcessor`** (SageMaker SDK v2.x) to:

1. **Write** `redaction_script.py` — the PySpark + Presidio script that runs *inside* the managed container
2. **Submit** a Processing Job — SageMaker provisions the instance, mounts the S3 input, runs the script, and copies the output back to S3 automatically
3. **Preview** the results directly from S3 — the raw clinical data never touches the notebook's local disk

Data flow: `S3 input` → *(SageMaker mounts)* → `container /opt/ml/processing/input` → Presidio PII detection + anonymisation → `container /opt/ml/processing/output` → *(SageMaker copies back)* → `S3 output`

**Key differences from Spark-NLP:**
- No assembly JAR to stage — Presidio is pure Python
- No pre-staged model files — `en_core_web_lg` is downloaded by the script at job startup
- Output parquet contains `redacted_aspects` and `redacted_suggestions` columns directly (detection + anonymisation happen in one step)
- ORGANIZATION entities are excluded from redaction, matching the original behaviour

In [11]:
%%writefile redaction_script.py
"""
Presidio PII redaction processing script.
Executed inside a SageMaker PySparkProcessor container via spark-submit.

No pre-staged models needed — Presidio and spaCy en_core_web_lg / fr_core_news_lg
are installed at job startup.  The container must have outbound internet access; for
VPC-only deployments, swap the spacy downloads for mounted S3 model packages.
"""
import os, sys, subprocess, glob, traceback

# ── Early diagnostics (always visible in CloudWatch) ─────────────────────────
print("=" * 60, flush=True)
print(f"Python  : {sys.version}", flush=True)
print(f"CWD     : {os.getcwd()}", flush=True)
print(f"PATH    : {os.environ.get('PATH','')}", flush=True)
print("=" * 60, flush=True)

# ── Install dependencies ───────────────────────────────────────────────────────
# Split into separate commands so a single failure doesn't block everything.
# spaCy is pinned to <3.8 — spaCy 3.8+ requires thinc>=8.3.12 which is not
# available in the sagemaker-spark-processing:3.3 container (Python 3.9).
# IMPORTANT: presidio and spaCy must be in the SAME pip command so the resolver
# sees the spaCy pin and doesn't pull in spaCy 3.8.x as a presidio dependency.
# pandas<2.0 is intentionally omitted; the container ships pandas 2.x which works.
_pip_cmds = [
    # Core data / excel deps — install first so xlsx reading always works
    [sys.executable, "-m", "pip", "install", "-q",
     "openpyxl", "langdetect"],
    # Presidio + spaCy in ONE command so the resolver honours the spaCy pin
    [sys.executable, "-m", "pip", "install", "-q",
     "presidio-analyzer>=2.2", "presidio-anonymizer>=2.2", "spacy>=3.4,<3.8"],
    # spaCy language models
    [sys.executable, "-m", "spacy", "download", "en_core_web_lg", "--quiet"],
    [sys.executable, "-m", "spacy", "download", "fr_core_news_lg", "--quiet"],
]

for cmd in _pip_cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    tag = " ".join(cmd[-4:])
    print(f"$ {tag} → exit {r.returncode}", flush=True)
    if r.returncode != 0:
        print(r.stderr[-2000:], flush=True)

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import udf
    from pyspark.sql.types import StringType
    import pyspark.sql.functions as F
    import pandas as pd
    print("Imports OK", flush=True)
    print(f"pandas  {pd.__version__}", flush=True)
except Exception:
    print("IMPORT ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

INPUT_DIR  = "/opt/ml/processing/input"
OUTPUT_DIR = "/opt/ml/processing/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    spark = SparkSession.builder.appName("Presidio-Redaction").getOrCreate()
    print(f"Spark {spark.version} started", flush=True)
except Exception:
    print("SPARKSESSION ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

# ── Read input ────────────────────────────────────────────────────────────────
xlsx_files = (
    glob.glob(os.path.join(INPUT_DIR, "**/*.xlsx"), recursive=True)
    + glob.glob(os.path.join(INPUT_DIR, "*.xlsx"))
)
if not xlsx_files:
    raise FileNotFoundError(f"No .xlsx files found under {INPUT_DIR}")

print(f"Reading: {xlsx_files[0]}", flush=True)
pdf = pd.read_excel(xlsx_files[0], dtype=str).fillna("")

# pandas 2.0 removed DataFrame.iteritems(); PySpark 3.3.x calls it internally
# when converting a pandas DataFrame to a Spark DataFrame.
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items

sdf = spark.createDataFrame(pdf)
print(f"Input  : {sdf.count()} rows × {len(sdf.columns)} columns", flush=True)

print("\nActual DataFrame columns:")
for idx, c in enumerate(sdf.columns):
    print(f"  [{idx:02d}] {repr(c)}")

# ── Locate target columns ─────────────────────────────────────────────────────
COL_ASPECTS     = "Aspects positifs de l'expérience globale"
COL_SUGGESTIONS = "Suggestions d'amélioration de l'expérience globale"

def find_column(df, target):
    if target in df.columns:
        return target
    for c in df.columns:
        if c.strip() == target.strip():
            print(f"  WARNING: matched '{c}' for target '{target}' after stripping")
            return c
    return None

actual_aspects     = find_column(sdf, COL_ASPECTS)
actual_suggestions = find_column(sdf, COL_SUGGESTIONS)

if actual_aspects is None:
    raise ValueError(f"Column not found: {repr(COL_ASPECTS)}\nAvailable: {sdf.columns}")
if actual_suggestions is None:
    raise ValueError(f"Column not found: {repr(COL_SUGGESTIONS)}\nAvailable: {sdf.columns}")

print(f"aspects     → {repr(actual_aspects)}")
print(f"suggestions → {repr(actual_suggestions)}")

# ── Presidio redaction UDF ────────────────────────────────────────────────────
# Entities to redact.  ORGANIZATION is intentionally excluded — mirrors the
# original Spark-NLP behaviour where ORG spans were left as-is.
ENTITIES = [
    "PERSON", "LOCATION", "DATE_TIME",
    "PHONE_NUMBER", "EMAIL_ADDRESS",
    "CREDIT_CARD", "IBAN_CODE",
    "IP_ADDRESS", "NRP",
]

# Module-level lazy singletons — initialised once per Spark executor process,
# not once per row, so the spaCy models are loaded only once per worker.
_analyzer   = None
_anonymizer = None

def _init_presidio():
    global _analyzer, _anonymizer
    if _analyzer is None:
        from presidio_analyzer import AnalyzerEngine
        from presidio_analyzer.nlp_engine import NlpEngineProvider
        from presidio_anonymizer import AnonymizerEngine
        provider = NlpEngineProvider(nlp_configuration={
            "nlp_engine_name": "spacy",
            "models": [
                {"lang_code": "en", "model_name": "en_core_web_lg"},
                {"lang_code": "fr", "model_name": "fr_core_news_lg"},
            ],
        })
        _analyzer   = AnalyzerEngine(
            nlp_engine=provider.create_engine(),
            supported_languages=["en", "fr"],
        )
        _anonymizer = AnonymizerEngine()

@udf(StringType())
def presidio_redact(text):
    """Detect and replace PII spans; returns redacted string with [ENTITY] tags."""
    if not text or not text.strip():
        return text
    _init_presidio()
    # Detect language; default to French (dataset is a French-language hospital survey)
    lang = "fr"
    if len(text.strip()) >= 20:
        try:
            from langdetect import detect
            detected = detect(text)
            if detected in ("en", "fr"):
                lang = detected
        except Exception:
            pass
    from presidio_anonymizer.entities import OperatorConfig
    operators = {
        entity: OperatorConfig("replace", {"new_value": f"[{entity}]"})
        for entity in ENTITIES
    }
    results  = _analyzer.analyze(text=text, entities=ENTITIES, language=lang)
    redacted = _anonymizer.anonymize(
        text=text, analyzer_results=results, operators=operators
    )
    return redacted.text

# ── Apply redaction to both columns ──────────────────────────────────────────
TEXT_COLS = {
    actual_aspects    : "redacted_aspects",
    actual_suggestions: "redacted_suggestions",
}

result_df = sdf
for src_col, out_col in TEXT_COLS.items():
    print(f"\n[Redact] '{src_col}' → '{out_col}'", flush=True)
    result_df = result_df.withColumn(out_col, presidio_redact(F.col(src_col)))

n = result_df.count()
print(f"\nRedaction complete — {n} rows", flush=True)

# ── Write output ──────────────────────────────────────────────────────────────
out_path = "file://" + os.path.join(OUTPUT_DIR, "redacted")
result_df.coalesce(1).write.mode("overwrite").parquet(out_path)
print(f"Saved to: {out_path}", flush=True)

spark.stop()

Overwriting redaction_script.py


In [12]:
# ── Resolve a SageMaker-assumable IAM execution role ──────────────────────────
import boto3, json, time

iam          = boto3.client("iam")
SM_ROLE_NAME = "SageMakerExecutionRole-Presidio"

TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect"   : "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action"   : "sts:AssumeRole",
    }],
})

try:
    sm_role = iam.get_role(RoleName=SM_ROLE_NAME)["Role"]["Arn"]
    print(f"Reusing existing role: {sm_role}")

except iam.exceptions.NoSuchEntityException:
    print(f"Creating '{SM_ROLE_NAME}' ...")
    sm_role = iam.create_role(
        RoleName                 = SM_ROLE_NAME,
        AssumeRolePolicyDocument = TRUST_POLICY,
        Description              = "SageMaker execution role for Presidio processing jobs",
    )["Role"]["Arn"]

    for policy_arn in [
        "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
        "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    ]:
        iam.attach_role_policy(RoleName=SM_ROLE_NAME, PolicyArn=policy_arn)
        print(f"  Attached {policy_arn}")

    print("Waiting 15 s for IAM propagation ...")
    time.sleep(15)

print(f"\nsm_role = {sm_role}")

Reusing existing role: arn:aws:iam::493644444178:role/SageMakerExecutionRole-Presidio

sm_role = arn:aws:iam::493644444178:role/SageMakerExecutionRole-Presidio


In [13]:
# ── Fix S3 keys: remove double-slashes and zero-byte folder markers ───────────
# SageMaker's data downloader rejects keys containing '//' (e.g.
# redacted_input/20260324_034447_parquet//part-00000.parquet).
# This cell copies any such objects to normalised paths and deletes the originals.

import boto3, re

_s3      = boto3.client("s3")
_prefix  = "spark-nlp/redaction/jgh-msss-prem/redacted_input/"
_pager   = _s3.get_paginator("list_objects_v2")

fixed = deleted = skipped = 0

for page in _pager.paginate(Bucket=bucket, Prefix=_prefix):
    for obj in page.get("Contents", []):
        key  = obj["Key"]
        size = obj["Size"]

        # Drop zero-byte folder-marker objects (key ends with '/')
        if key.endswith("/") and size == 0:
            _s3.delete_object(Bucket=bucket, Key=key)
            print(f"  [deleted folder marker] {key}")
            deleted += 1
            continue

        # Fix keys with '//' in them
        clean_key = re.sub(r"/+", "/", key)
        if clean_key != key:
            _s3.copy_object(
                Bucket=bucket, Key=clean_key,
                CopySource={"Bucket": bucket, "Key": key},
            )
            _s3.delete_object(Bucket=bucket, Key=key)
            print(f"  [fixed] {key}\n      → {clean_key}")
            fixed += 1
        else:
            skipped += 1

print(f"\nDone — fixed: {fixed}, folder markers deleted: {deleted}, already clean: {skipped}")


Done — fixed: 0, folder markers deleted: 0, already clean: 3


In [14]:
# ── Submit Presidio job to SageMaker ─────────────────────────────────────────
# No assembly JAR and no pre-staged model files are needed.
# Presidio + spaCy en_core_web_lg / fr_core_news_lg are pip-installed by the script at startup.
from sagemaker.spark.processing import PySparkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

s3_input_path  = f"s3://{bucket}/spark-nlp/redaction/jgh-msss-prem/redacted_input"
s3_output_path = f"s3://{bucket}/presidio/redaction/jgh-msss-prem/redacted_output"

spark_processor = PySparkProcessor(
    base_job_name          = "presidio-redaction",
    framework_version      = "3.3",
    role                   = sm_role,
    instance_count         = 1,
    instance_type          = "ml.m5.4xlarge",  # 16 vCPU / 64 GB
    max_runtime_in_seconds = 3600,
    sagemaker_session      = sm_session,
)

spark_processor.run(
    submit_app    = "redaction_script.py",
    # No submit_jars — Presidio is pure Python, no JVM assembly needed
    configuration = [{
        "Classification": "spark-defaults",
        "Properties": {
            "spark.driver.memory"           : "16G",
            "spark.driver.maxResultSize"    : "4000M",
            "spark.executor.memory"         : "20G",
            "spark.executor.memoryOverhead" : "4G",
        },
    }],
    inputs = [
        ProcessingInput(
            source      = s3_input_path,
            destination = "/opt/ml/processing/input",
        ),
    ],
    outputs = [
        ProcessingOutput(
            source      = "/opt/ml/processing/output",
            destination = s3_output_path,
        )
    ],
    logs = True,
    wait = True,
)

print(f"\nJob complete. Results at: {s3_output_path}")

INFO:sagemaker:Creating processing-job with name presidio-redaction-2026-03-31-17-07-25-108


............03-31 17:09 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '/opt/ml/processing/input/code/redaction_script.py']
03-31 17:09 smspark.cli  INFO     Raw spark options before processing: {'class_': None, 'jars': None, 'py_files': None, 'files': None, 'verbose': False}
03-31 17:09 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/redaction_script.py']
03-31 17:09 smspark.cli  INFO     Rendered spark options: {'class_': None, 'jars': None, 'py_files': None, 'files': None, 'verbose': False}
03-31 17:09 smspark.cli  INFO     Initializing processing job.
03-31 17:09 smspark-submit INFO     {'current_host': 'algo-1', 'current_instance_type': 'ml.m5.4xlarge', 'current_group_name': 'homogeneousCluster', 'hosts': ['algo-1'], 'instance_groups': [{'instance_group_name': 'homogeneousCluster', 'instance_type': 'ml.m5.4xlarge', 'hosts': ['algo-1']}], 'network_interface_name': 'eth0', 'topology': None}
03-31 17:09 smspark-submit INF

In [15]:
# ── Fetch ALL CloudWatch logs for the last Processing Job ─────────────────────
import boto3, time

logs      = boto3.client("logs", region_name=region)
job_name  = spark_processor.latest_job.job_name
log_group = "/aws/sagemaker/ProcessingJobs"

print(f"Job      : {job_name}")
print(f"LogGroup : {log_group}\n")

for attempt in range(10):
    resp    = logs.describe_log_streams(logGroupName=log_group, logStreamNamePrefix=job_name)
    streams = resp.get("logStreams", [])
    if streams:
        break
    print(f"Waiting for log streams... ({attempt+1}/10)")
    time.sleep(5)

if not streams:
    print("No log streams found.")
else:
    for stream in streams:
        name = stream["logStreamName"]
        print(f"\n{'='*70}")
        print(f"STREAM: {name}")
        print("=" * 70)

        all_events = []
        kwargs = dict(logGroupName=log_group, logStreamName=name, startFromHead=True)
        while True:
            resp2  = logs.get_log_events(**kwargs)
            events = resp2.get("events", [])
            if not events:
                break
            all_events.extend(events)
            token = resp2.get("nextForwardToken")
            if token == kwargs.get("nextToken"):
                break
            kwargs["nextToken"] = token

        print(f"(total {len(all_events)} log events)")
        for ev in all_events:
            print(ev["message"])

Job      : presidio-redaction-2026-03-31-17-07-25-108
LogGroup : /aws/sagemaker/ProcessingJobs


STREAM: presidio-redaction-2026-03-31-17-07-25-108/algo-1-1774976880
(total 1898 log events)
03-31 17:09 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '/opt/ml/processing/input/code/redaction_script.py']
03-31 17:09 smspark.cli  INFO     Raw spark options before processing: {'class_': None, 'jars': None, 'py_files': None, 'files': None, 'verbose': False}
03-31 17:09 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/redaction_script.py']
03-31 17:09 smspark.cli  INFO     Rendered spark options: {'class_': None, 'jars': None, 'py_files': None, 'files': None, 'verbose': False}
03-31 17:09 smspark.cli  INFO     Initializing processing job.
03-31 17:09 smspark-submit INFO     {'current_host': 'algo-1', 'current_instance_type': 'ml.m5.4xlarge', 'current_group_name': 'homogeneousCluster', 'hosts': ['algo-1'], 'instance_groups': [{'insta

In [16]:
# ── Preview results from S3 (no local download of raw input) ──────────────────
import io, boto3, pandas as pd

parsed_base = s3_output_path.replace("s3://", "").split("/", 1)
s3_bucket   = parsed_base[0]
s3_prefix   = parsed_base[1].rstrip("/") + "/"

paginator    = boto3.client("s3").get_paginator("list_objects_v2")
output_files = [
    f"s3://{s3_bucket}/{obj['Key']}"
    for page in paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix)
    for obj in page.get("Contents", [])
]
print(f"Output files ({len(output_files)}):")
for f in output_files:
    print(f"  {f}")

parquet_files = [f for f in output_files if f.endswith(".parquet")]
if parquet_files:
    parsed     = parquet_files[0].replace("s3://", "").split("/", 1)
    obj        = boto3.client("s3").get_object(Bucket=parsed[0], Key=parsed[1])
    preview_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    print(f"\nOutput preview  ({len(preview_df)} rows × {len(preview_df.columns)} cols):")
    pd.set_option("display.max_colwidth", 120)
    print(preview_df.head(5).to_string(index=False))
else:
    print("\nNo parquet output found yet — run the processor cell first.")

Output files (4):
  s3://sagemaker-us-west-2-493644444178/presidio/redaction/jgh-msss-prem/redacted_output/redacted/._SUCCESS.crc
  s3://sagemaker-us-west-2-493644444178/presidio/redaction/jgh-msss-prem/redacted_output/redacted/.part-00000-481c9da3-abc2-4164-9106-09e01aa56ac4-c000.snappy.parquet.crc
  s3://sagemaker-us-west-2-493644444178/presidio/redaction/jgh-msss-prem/redacted_output/redacted/_SUCCESS
  s3://sagemaker-us-west-2-493644444178/presidio/redaction/jgh-msss-prem/redacted_output/redacted/part-00000-481c9da3-abc2-4164-9106-09e01aa56ac4-c000.snappy.parquet

Output preview  (6761 rows × 7 cols):
Unnamed: 0               Date de complétion Évaluation                                                                                  Aspects positifs de l'expérience globale  Suggestions d'amélioration de l'expérience globale                                                                                                            redacted_aspects    redacted_suggestions
         0

In [17]:
# ── Download parquet output from S3 and load into local Spark session ─────────
import os, shutil, boto3

local_dir = "/tmp/redacted_output"
if os.path.exists(local_dir):
    shutil.rmtree(local_dir)
os.makedirs(local_dir)

parsed_base = s3_output_path.replace("s3://", "").split("/", 1)
s3_bucket   = parsed_base[0]
s3_prefix   = parsed_base[1].rstrip("/") + "/"

s3_client    = boto3.client("s3")
paginator    = s3_client.get_paginator("list_objects_v2")
all_files    = [
    obj["Key"]
    for page in paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix)
    for obj in page.get("Contents", [])
]
parquet_keys = [k for k in all_files if k.endswith(".parquet")]

print(f"Downloading {len(parquet_keys)} parquet file(s) to {local_dir} ...")
for key in parquet_keys:
    fname = os.path.basename(key)
    s3_client.download_file(s3_bucket, key, f"{local_dir}/{fname}")
    print(f"  {fname}")

result_sdf = spark.read.parquet(local_dir)
result_sdf = result_sdf.toDF(*[c.strip() for c in result_sdf.columns])
print(f"\nLoaded  : {result_sdf.count()} rows × {len(result_sdf.columns)} columns")
result_sdf.printSchema()

  part-00000-481c9da3-abc2-4164-9106-09e01aa56ac4-c000.snappy.parquet



Loaded  : 6761 rows × 7 columns
root
 |-- Unnamed: 0: string (nullable = true)
 |-- Date de complétion: string (nullable = true)
 |-- Évaluation: string (nullable = true)
 |-- Aspects positifs de l'expérience globale: string (nullable = true)
 |-- Suggestions d'amélioration de l'expérience globale: string (nullable = true)
 |-- redacted_aspects: string (nullable = true)
 |-- redacted_suggestions: string (nullable = true)



In [18]:
# ── Filter to English-only rows using langdetect ──────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langdetect"], check=True)

from langdetect import detect, LangDetectException
import pyspark.sql.functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

COL_ASPECTS     = "Aspects positifs de l'expérience globale"
COL_SUGGESTIONS = "Suggestions d'amélioration de l'expérience globale"

@udf(StringType())
def detect_lang(text):
    if not text or len(text.strip()) < 10:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

english_sdf = (
    result_sdf
    .withColumn("lang_aspects",     detect_lang(F.col(COL_ASPECTS)))
    .withColumn("lang_suggestions", detect_lang(F.col(COL_SUGGESTIONS)))
    .filter(
        (F.col("lang_aspects") == "en") | (F.col("lang_suggestions") == "en")
    )
    .drop("lang_aspects", "lang_suggestions")
)

total   = result_sdf.count()
english = english_sdf.count()
print(f"Total rows   : {total}")
print(f"English rows : {english}  ({english/total*100:.1f}%)")

Total rows   : 6761
English rows : 2868  (42.4%)


In [19]:
# ── Display redacted results, insert redacted cols in-place, upload ───────────
# The processing script already produced redacted_aspects and
# redacted_suggestions — no further UDF redaction step needed here.
import os
from sagemaker.s3 import S3Uploader

# Build column order: insert each redacted column immediately after its source
base_cols = [c for c in result_sdf.columns
             if c not in ("redacted_aspects", "redacted_suggestions")]
ordered_cols = []
for c in base_cols:
    ordered_cols.append(c)
    if c == COL_ASPECTS:
        ordered_cols.append("redacted_aspects")
    elif c == COL_SUGGESTIONS:
        ordered_cols.append("redacted_suggestions")

final_sdf = english_sdf.select(*ordered_cols)
print(f"Final table: {final_sdf.count()} English rows × {len(ordered_cols)} columns\n")
final_sdf.show(truncate=120)

# ── Upload to S3 ──────────────────────────────────────────────────────────────
local_out  = "/tmp/redacted_ana"
os.makedirs(local_out, exist_ok=True)
local_file = f"{local_out}/redacted_ana.parquet"

final_sdf.toPandas().to_parquet(local_file, index=False)
print(f"Written locally: {local_file}")

target_s3 = f"s3://{bucket}/presidio/redaction/jgh-msss-prem/redacted_output/redacted_ana"
S3Uploader.upload(local_out, target_s3, sagemaker_session=sm_session)
print(f"Uploaded to    : {target_s3}")

Final table: 2871 English rows × 7 columns



+----------+--------------------------------+----------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+
|Unnamed: 0|              Date de complétion|Évaluation|                                                                                Aspects positifs de l'expérience globale|                                                                                                        redacted_aspects|                                                                      Suggestions d'amélioration de l'expérience globale|                                        

Written locally: /tmp/redacted_ana/redacted_ana.parquet
Uploaded to    : s3://sagemaker-us-west-2-493644444178/presidio/redaction/jgh-msss-prem/redacted_output/redacted_ana
